In [2]:
!unzip /content/processed_children_story_data_v2.zip -d /content/anuj232003_nlp_project_dataset_path

Archive:  /content/processed_children_story_data_v2.zip
  inflating: /content/anuj232003_nlp_project_dataset_path/train.jsonl  
  inflating: /content/anuj232003_nlp_project_dataset_path/rejected_rows.csv  
  inflating: /content/anuj232003_nlp_project_dataset_path/suspicious_accepted_rows.csv  
  inflating: /content/anuj232003_nlp_project_dataset_path/test.jsonl  
  inflating: /content/anuj232003_nlp_project_dataset_path/val.jsonl  
  inflating: /content/anuj232003_nlp_project_dataset_path/accepted_metadata.csv  
  inflating: /content/anuj232003_nlp_project_dataset_path/sample_examples.jsonl  
  inflating: /content/anuj232003_nlp_project_dataset_path/all_processed.jsonl  


In [3]:
anuj232003_nlp_project_dataset_path = 'anuj232003/nlp-project-dataset'
anuj232003_nlp_checkpoint_path = 'anuj232003/nlp-checkpoint'

print('Data source import complete.')


Data source import complete.


In [4]:
pip install -U transformers peft trl datasets accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: p

In [7]:
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# import json
# import gc
# import torch

# from datasets import load_dataset
# from transformers import (
#     AutoTokenizer,
#     AutoModelForCausalLM,
#     BitsAndBytesConfig,
#     set_seed,
# )
# from peft import LoraConfig, prepare_model_for_kbit_training
# from trl import SFTTrainer, SFTConfig

# # =========================================================
# # Clean memory
# # =========================================================

# gc.collect()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()

# # =========================================================
# # Config
# # =========================================================

# MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
# DATA_DIR = "/content/anuj232003_nlp_project_dataset_path"

# TRAIN_FILE = os.path.join(DATA_DIR, "train.jsonl")
# VAL_FILE = os.path.join(DATA_DIR, "val.jsonl")
# TEST_FILE = os.path.join(DATA_DIR, "test.jsonl")

# OUTPUT_DIR = "mistral_children_story_lora_ckpt"
# SEED = 42

# # Safer Kaggle settings
# MAX_SEQ_LENGTH = 512

# # LoRA
# LORA_R = 4
# LORA_ALPHA = 8
# LORA_DROPOUT = 0.05
# LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# # Training
# NUM_EPOCHS = 1
# LEARNING_RATE = 2e-5
# TRAIN_BATCH_SIZE = 1
# EVAL_BATCH_SIZE = 1
# GRAD_ACCUM_STEPS = 16
# WARMUP_RATIO = 0.03
# WEIGHT_DECAY = 0.01
# LOGGING_STEPS = 10
# EVAL_STEPS = 100
# SAVE_STEPS = 100
# SAVE_TOTAL_LIMIT = 3

# SYSTEM_PROMPT = """You are a children's story writer.

# Write safe, warm, easy-to-understand stories for children ages 6 to 8.
# Follow the requested format exactly.
# """

# set_seed(SEED)

# # =========================================================
# # Tokenizer
# # =========================================================

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "right"

# # =========================================================
# # 4-bit quantization config
# # =========================================================

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=torch.float16,
# )

# # =========================================================
# # Model
# # =========================================================

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     quantization_config=bnb_config,
#     device_map={"": 0},
#     low_cpu_mem_usage=True,
# )

# model.config.use_cache = False
# model.config.pretraining_tp = 1
# model.gradient_checkpointing_enable()

# # Important for QLoRA
# model = prepare_model_for_kbit_training(model)

# # =========================================================
# # Dataset
# # =========================================================

# dataset = load_dataset(
#     "json",
#     data_files={
#         "train": TRAIN_FILE,
#         "validation": VAL_FILE,
#         "test": TEST_FILE,
#     },
# )

# print(dataset)
# print("\nSample raw row:")
# print(dataset["train"][0])

# def to_conversational_prompt_completion(example):
#     instruction = example["instruction"].strip()
#     user_input = example["input"].strip()
#     output = example["output"].strip()

#     return {
#         "prompt": [
#             {
#                 "role": "system",
#                 "content": SYSTEM_PROMPT,
#             },
#             {
#                 "role": "user",
#                 "content": f"{instruction}\n\n{user_input}",
#             },
#         ],
#         "completion": [
#             {
#                 "role": "assistant",
#                 "content": output,
#             }
#         ],
#     }

# dataset = dataset.map(
#     to_conversational_prompt_completion,
#     remove_columns=dataset["train"].column_names,
# )

# print("\nConverted dataset:")
# print(dataset)
# print("\nSample converted row:")
# print(dataset["train"][0])

# # =========================================================
# # LoRA config
# # =========================================================

# peft_config = LoraConfig(
#     r=LORA_R,
#     lora_alpha=LORA_ALPHA,
#     lora_dropout=LORA_DROPOUT,
#     bias="none",
#     task_type="CAUSAL_LM",
#     target_modules=LORA_TARGET_MODULES,
# )

# # =========================================================
# # Training config
# # =========================================================

# training_args = SFTConfig(
#     output_dir=OUTPUT_DIR,
#     max_length=MAX_SEQ_LENGTH,
#     learning_rate=LEARNING_RATE,
#     num_train_epochs=NUM_EPOCHS,
#     per_device_train_batch_size=TRAIN_BATCH_SIZE,
#     per_device_eval_batch_size=EVAL_BATCH_SIZE,
#     gradient_accumulation_steps=GRAD_ACCUM_STEPS,
#     warmup_ratio=WARMUP_RATIO,
#     weight_decay=WEIGHT_DECAY,
#     logging_steps=LOGGING_STEPS,
#     eval_strategy="steps",
#     eval_steps=EVAL_STEPS,
#     save_strategy="steps",
#     save_steps=SAVE_STEPS,
#     save_total_limit=SAVE_TOTAL_LIMIT,
#     bf16=False,
#     fp16=False,
#     lr_scheduler_type="cosine",
#     report_to="none",
#     seed=SEED,
#     packing=False,
#     max_grad_norm=0.5,
#     completion_only_loss=True,
#     optim="paged_adamw_8bit",
#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",
#     greater_is_better=False,
# )

# print("\n===== DEBUG SETTINGS =====")
# print("bnb compute dtype:", bnb_config.bnb_4bit_compute_dtype)
# print("training fp16:", training_args.fp16)
# print("training bf16:", training_args.bf16)
# print("max_length:", training_args.max_length)
# print("learning_rate:", training_args.learning_rate)
# print("train batch size:", training_args.per_device_train_batch_size)
# print("eval batch size:", training_args.per_device_eval_batch_size)
# print("grad_accum_steps:", training_args.gradient_accumulation_steps)
# print("eval_steps:", training_args.eval_steps)
# print("save_steps:", training_args.save_steps)
# print("==========================\n")

# # =========================================================
# # Trainer
# # =========================================================

# trainer = SFTTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=dataset["train"],
#     eval_dataset=dataset["validation"],
#     peft_config=peft_config,
# )

# # =========================================================
# # Train
# # =========================================================

# print("\nStarting training...")
# train_result = trainer.train()

# # =========================================================
# # Save final model
# # =========================================================

# print("\nSaving adapter + tokenizer...")
# trainer.save_model(OUTPUT_DIR)
# tokenizer.save_pretrained(OUTPUT_DIR)

# # Save train metrics
# train_metrics = train_result.metrics
# with open(os.path.join(OUTPUT_DIR, "train_metrics.json"), "w") as f:
#     json.dump(train_metrics, f, indent=2)

# # Save trainer log history
# with open(os.path.join(OUTPUT_DIR, "trainer_log_history.json"), "w") as f:
#     json.dump(trainer.state.log_history, f, indent=2)

# # =========================================================
# # Evaluate after training
# # =========================================================

# print("\nEvaluating best/final model on validation set...")
# val_metrics = trainer.evaluate(dataset["validation"])
# print(val_metrics)

# print("\nEvaluating best/final model on test set...")
# test_metrics = trainer.evaluate(dataset["test"])
# print(test_metrics)

# with open(os.path.join(OUTPUT_DIR, "val_metrics.json"), "w") as f:
#     json.dump(val_metrics, f, indent=2)

# with open(os.path.join(OUTPUT_DIR, "test_metrics.json"), "w") as f:
#     json.dump(test_metrics, f, indent=2)

# print(f"\nDone. Outputs saved to: {OUTPUT_DIR}")

In [ ]:
# !zip -r submission.zip /kaggle/working/mistral_children_story_lora_ckpt/checkpoint-100


In [ ]:
# !zip -r output_files.zip /kaggle/working/mistral_children_story_lora_ckpt/checkpoint-100


In [ ]:
# from IPython.display import FileLink
# FileLink(r'output_files.zip') # Use the name of your zip file

In [10]:
!mkdir -p /content/anuj232003/nlp-checkpoint/kaggle/working/mistral_children_story_lora_ckpt
!unzip /content/Checkpoint_100.zip -d /content/anuj232003/nlp-checkpoint/kaggle/working/mistral_children_story_lora_ckpt/checkpoint-100

Archive:  /content/Checkpoint_100.zip
   creating: /content/anuj232003/nlp-checkpoint/kaggle/working/mistral_children_story_lora_ckpt/checkpoint-100/Checkpoint_100/
  inflating: /content/anuj232003/nlp-checkpoint/kaggle/working/mistral_children_story_lora_ckpt/checkpoint-100/Checkpoint_100/adapter_model.safetensors  
  inflating: /content/anuj232003/nlp-checkpoint/kaggle/working/mistral_children_story_lora_ckpt/checkpoint-100/__MACOSX/Checkpoint_100/._adapter_model.safetensors  
  inflating: /content/anuj232003/nlp-checkpoint/kaggle/working/mistral_children_story_lora_ckpt/checkpoint-100/Checkpoint_100/tokenizer_config.json  
  inflating: /content/anuj232003/nlp-checkpoint/kaggle/working/mistral_children_story_lora_ckpt/checkpoint-100/__MACOSX/Checkpoint_100/._tokenizer_config.json  
  inflating: /content/anuj232003/nlp-checkpoint/kaggle/working/mistral_children_story_lora_ckpt/checkpoint-100/Checkpoint_100/optimizer.pt  
  inflating: /content/anuj232003/nlp-checkpoint/kaggle/working/m

In [8]:
pip install -U bitsandbytes>=0.46.1

In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# =========================================================
# Paths
# =========================================================

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
ADAPTER_PATH = "/content/anuj232003/nlp-checkpoint/kaggle/working/mistral_children_story_lora_ckpt/checkpoint-100/Checkpoint_100"   # change this if needed

# =========================================================
# Quantization config
# =========================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# =========================================================
# Load tokenizer
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# =========================================================
# Load base model
# =========================================================

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

# =========================================================
# Attach LoRA adapter
# =========================================================

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

# =========================================================
# Prompt template
# =========================================================

SYSTEM_PROMPT = """You are a children's story writer.

Write safe, warm, easy-to-understand stories for children ages 6 to 8.
Follow the requested format exactly.
"""

INSTRUCTION = """Write a children's story for ages 6 to 8.

Requirements:
- Use simple vocabulary and clear sentences
- Keep the story under 500 words
- Use 2 to 3 main characters
- Make the story suitable for 6 to 7 visual scenes
- Include a clear beginning, middle, and end
- End with an explicit moral lesson
- Avoid scary, violent, dark, or adult themes

Output format:
Title: ...
Characters: ...
Story:
...
Moral: ...
"""

def generate_story(theme: str, max_new_tokens: int = 400):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{INSTRUCTION}\n\nTheme: {theme}"},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.08,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)
    return text.strip()

# =========================================================
# Test on a few themes
# =========================================================

themes = ["kindness", "honesty", "sharing", "patience", "teamwork"]

for theme in themes:
    print("\n" + "=" * 80)
    print(f"THEME: {theme}")
    print("=" * 80)
    print(generate_story(theme))

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


THEME: kindness
Title: Timmy and Sammy
Characters: Timmy (a boy), Sammy (a dog)
Story: Once upon a time, in a small town named Meadowville lived a young boy named Timmy. He was a friendly boy who loved animals more than anything else. One sunny day while playing near his house, he saw a poor, scruffy little dog named Sammy. Timmy felt sad seeing Sammy alone and sad. He approached him and asked, "Hi there, friend! Why are you all alone?" Sammy looked up at Timmy and wagged his tail, as if thanking him for noticing him. Timmy took pity on Sammy and decided to take him home. After asking his parents, they agreed to keep Sammy as their pet. Timmy and Sammy became best friends soon after that day. They played together every day and even went on small adventures around their neighborhood. People began to notice how happy Timmy seemed and started asking about his new friend. Timmy proudly told everyone that this was Sammy. Everyone was amazed at how Timmy transformed Sammy from being a scare

In [13]:
theme = "kindness"
output = generate_story(theme)
print(output)

Title: Sammy and Spike
Characters: Sammy, Spike
Story:
Once upon a time in a quiet neighborhood lived two best friends named Sammy and Spike. They were always together and loved playing games, sharing secrets, and laughing together. But one day, something changed between them.

One sunny afternoon while they played hide-and-seek in their favorite hiding spot, Spike found a shiny toy car that he hid beforehand and forgot about it. When Sammy discovered this treasure, he felt sad because Spike didn’t share it with him even though they shared everything. Feeling hurt, Sammy decided not to play with Spike anymore and went home alone.

Soon enough, Spike missed his buddy and realized what a mistake he had made. He searched high and low for Sammy but couldn't find him. Suddenly, he remembered how much fun they had building sandcastles at the beach. So, Spike packed a picnic basket filled with sandwiches, juice boxes, cookies, and toys and headed off on a adventure.

When he arrived at the be

In [18]:
import os
import gc
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# =========================================================
# Config
# =========================================================

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
ADAPTER_PATH = "/content/anuj232003/nlp-checkpoint/kaggle/working/mistral_children_story_lora_ckpt/checkpoint-100/Checkpoint_100"

THEMES = ["kindness", "honesty"]

GEN_CONFIG = {
    "max_new_tokens": 300,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,
    "repetition_penalty": 1.08,
}

# =========================================================
# Helpers
# =========================================================

def clear_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# =========================================================
# Load tokenizer
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SYSTEM_PROMPT = """You are a children's story writer.

Write safe, warm, easy-to-understand stories for children ages 6 to 8.
Follow the requested format exactly.
"""

INSTRUCTION = """Write a children's story for ages 6 to 8.

Requirements:
- Use simple vocabulary and clear sentences
- Keep the story under 500 words
- Use 2 to 3 main characters
- Make the story suitable for 6 to 7 visual scenes
- Include a clear beginning, middle, and end
- End with an explicit moral lesson
- Avoid scary, violent, dark, or adult themes

Output format:
Title: ...
Characters: ...
Story:
...
Moral: ...
"""

FEW_SHOT_EXAMPLES = [
    {
        "theme": "sharing",
        "output": """Title: Mira and the Mango Basket

Characters: Mira, Rafi, Grandma

Story:
Mira loved carrying Grandma's basket of mangoes from the garden to the kitchen.
One sunny morning, Grandma filled the basket with golden mangoes and asked Mira to help.
Mira smiled proudly and lifted the basket with both hands.

On the way home, Mira met her friend Rafi near the gate.
Rafi had been helping his father sweep leaves, and he looked hot and tired.
He stared at the basket and said, "Those mangoes look sweet."

Mira remembered how much she loved mangoes.
She wanted to keep all of them for her family.
For a moment, she held the basket a little tighter and kept walking.

But then she noticed Rafi wiping his forehead.
She stopped and thought about how Grandma always shared what they had.
Mira picked the ripest mango and handed it to Rafi with a smile.

Rafi's face lit up.
"Thank you, Mira!" he said.
They sat under the tree and talked while Rafi ate the mango.
When Mira reached home, Grandma asked why one mango was missing.

Mira told her the truth.
Grandma smiled warmly and said, "A shared fruit becomes sweeter."
Mira felt happy inside.
That evening, the whole family ate mango slices together, and Mira knew she had done the right thing.

Moral: Sharing what you have can make everyone happier."""
    },
    {
        "theme": "patience",
        "output": """Title: Niko and the Little Seed

Characters: Niko, Mama, Pip the sparrow

Story:
Niko found a tiny seed near the garden fence and ran to show Mama.
"Can we plant it today?" he asked.
Mama nodded, and together they placed the seed in soft brown soil.

The next morning, Niko rushed outside.
He looked at the pot and frowned.
Nothing had grown.
He poked the soil gently and sighed, "Why is it taking so long?"

Pip the sparrow landed on the fence and chirped.
Niko came back again at lunch, and then again before dinner.
Still nothing happened.
Niko crossed his arms and said, "Maybe this seed does not work."

Mama knelt beside him and said, "Some good things need time."
She helped him water the soil carefully.
The next day, Niko waited again.
This time he did not poke the dirt.
He only watered it and placed the pot where the sun could shine.

A few days later, a tiny green sprout peeked out.
Niko clapped and laughed.
Each morning, he cared for the plant a little more.
Soon it grew strong and tall.

Niko smiled at Pip and said, "I am glad I did not give up."
Mama hugged him and said, "Patience helps good things grow."
Niko looked proudly at the little plant dancing in the breeze.

Moral: Good things often take time, so patience is important."""
    },
]

def build_zero_shot_messages(theme: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{INSTRUCTION}\n\nTheme: {theme}"},
    ]

def build_few_shot_messages(theme: str):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for ex in FEW_SHOT_EXAMPLES:
        messages.append({"role": "user", "content": f"{INSTRUCTION}\n\nTheme: {ex['theme']}"})
        messages.append({"role": "assistant", "content": ex["output"]})
    messages.append({"role": "user", "content": f"{INSTRUCTION}\n\nTheme: {theme}"})
    return messages

def generate(model, messages):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **GEN_CONFIG,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# =========================================================
# Phase 1: Base model only
# =========================================================

clear_mem()
print("\nLoading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={"": 0},   # force single GPU only
    low_cpu_mem_usage=True,
)

base_outputs = {}

for theme in THEMES:
    print("\n" + "=" * 100)
    print(f"THEME: {theme}")
    print("=" * 100)

    zero_out = generate(base_model, build_zero_shot_messages(theme))
    few_out = generate(base_model, build_few_shot_messages(theme))

    print("\n--- BASE ZERO-SHOT ---\n")
    print(zero_out)
    print("\n--- BASE FEW-SHOT ---\n")
    print(few_out)

    base_outputs[theme] = {
        "base_zero_shot": zero_out,
        "base_few_shot": few_out,
    }

# free memory before loading LoRA
del base_model
clear_mem()

# =========================================================
# Phase 2: Base + LoRA adapter
# =========================================================

print("\nLoading base model again for LoRA...")
base_model_for_lora = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

print("Loading LoRA adapter...")
lora_model = PeftModel.from_pretrained(base_model_for_lora, ADAPTER_PATH)
lora_model.eval()

rows = []

for theme in THEMES:
    lora_out = generate(lora_model, build_zero_shot_messages(theme))

    print("\n" + "=" * 100)
    print(f"THEME: {theme}")
    print("=" * 100)
    print("\n--- LORA CHECKPOINT-100 ---\n")
    print(lora_out)

    rows.append({
        "theme": theme,
        "base_zero_shot": base_outputs[theme]["base_zero_shot"],
        "base_few_shot": base_outputs[theme]["base_few_shot"],
        "lora_checkpoint_100": lora_out,
    })

df = pd.DataFrame(rows)
df.to_csv("/content/story_model_comparison.csv", index=False)
print("\nSaved comparison to /content/story_model_comparison.csv")

CUDA available: True
GPU: Tesla T4

Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


THEME: kindness

--- BASE ZERO-SHOT ---

Title: The Kind Kangaroo's Adventure

Characters:
1. Kanga, a kind and gentle kangaroo
2. Roo, Kanga's curious baby joey
3. Grouchy Goanna, a grumpy lizard

Story:
In the sunny land of Australia, lived a kind and gentle kangaroo named Kanga. She had a tiny baby joey named Roo who loved hopping around and exploring his world. Kanga always made sure that Roo was safe and happy.

One sunny morning, while they were out for a walk, Roo saw some beautiful flowers in a meadow. He hopped towards them, leaving Kanga behind. As he reached the meadow, he met Grouchy Goanna, a grumpy lizard who lived there.

Grouchy Goanna didn't like little Roo because he thought that all young animals were noisy and disruptive. "Go away, little pest!" he hissed. But Roo wasn't afraid. Instead, he picked up a lovely flower and gave it to Goanna as a gift.

Goanna was taken aback by Roo's kindness. He had never received a gift from a young animal before. The kindness of th

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading LoRA adapter...

THEME: kindness

--- LORA CHECKPOINT-100 ---

Title: Sammy and Oliver
Characters: Sammy, Oliver
Story:
Once upon a time in a beautiful green town named Sunville lived two best friends named Sammy and Oliver. They were always together, playing hide-and-seek, building sandcastles at the beach, and picking juicy ripe apples from their trees. But there was something special about these two friends - they loved to share!

One sunny day as they played in the park, something amazing happened. A little bird called Oliver had lost his home in a storm. He was shivering and hopping from one foot to another, feeling sad and all alone. As he chirped out cries of despair, Sammy the big elephant walked up to him. "Hello little friend," said Sammy kindly, "Why do you look so worried?" Oliver explained how he had lost his nest. Hearing this sad news, Sammy didn't say a word but took Oliver on his strong back and carried him to find a new home.

They searched high and low till t